In [1]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.0"

import keras.backend as K

from keras.utils.vis_utils import plot_model

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)
tf.random.set_seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

In [2]:
# Try to make GPU memory allocation dynamic, so it doesn't slow down/crash
physical_devices = tf.config.list_physical_devices('GPU')
try:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)
    # config = tf.compat.v1.ConfigProto()
    # config.gpu_options.allow_growth = True
    # session = tf.compat.v1.Session(config=config)
    tf.debugging.set_log_device_placement(False)
except:
    print('oh no.')
  # Invalid device or cannot modify virtual devices once initialized.
    pass

In [3]:
pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install imutils

Note: you may need to restart the kernel to use updated packages.


In [5]:
pip install visualkeras

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install pydot

Note: you may need to restart the kernel to use updated packages.


In [7]:
from tqdm import tqdm
import cv2
# import os
import imutils

# change this to wherever your images are located
DATASET_PATH = r'C:/Users/Aihara/Desktop/Temporary Jupter Stuff/Kaggle - Brain Tumor MRI Dataset'


def crop_img(img):
	"""
	Finds the extreme points on the image and crops the rectangular out of them
	"""
	gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
	gray = cv2.GaussianBlur(gray, (3, 3), 0)

	# threshold the image, then perform a series of erosions +
	# dilations to remove any small regions of noise
	thresh = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
	thresh = cv2.erode(thresh, None, iterations=2)
	thresh = cv2.dilate(thresh, None, iterations=2)

	# find contours in thresholded image, then grab the largest one
	cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
	cnts = imutils.grab_contours(cnts)
	c = max(cnts, key=cv2.contourArea)

	# find the extreme points
	extLeft = tuple(c[c[:, :, 0].argmin()][0])
	extRight = tuple(c[c[:, :, 0].argmax()][0])
	extTop = tuple(c[c[:, :, 1].argmin()][0])
	extBot = tuple(c[c[:, :, 1].argmax()][0])
	ADD_PIXELS = 0
	new_img = img[extTop[1]-ADD_PIXELS:extBot[1]+ADD_PIXELS, extLeft[0]-ADD_PIXELS:extRight[0]+ADD_PIXELS].copy()
	
	return new_img
	
if __name__ == "__main__":
	training = DATASET_PATH + "/Training"
	testing = DATASET_PATH + "/Testing"
	training_dir = os.listdir(training)
	testing_dir = os.listdir(testing)
	IMG_SIZE = 256

	for dir in training_dir:
		save_path = 'cleaned/Training/'+ dir
		path = os.path.join(training,dir)
		image_dir = os.listdir(path)
		for img in image_dir:
			image = cv2.imread(os.path.join(path,img))
			new_img = crop_img(image)
			new_img = cv2.resize(new_img,(IMG_SIZE,IMG_SIZE))
			if not os.path.exists(save_path):
				os.makedirs(save_path)
			cv2.imwrite(save_path+'/'+img, new_img)
	
	for dir in testing_dir:
		save_path = 'cleaned/Testing/'+ dir
		path = os.path.join(testing,dir)
		image_dir = os.listdir(path)
		for img in image_dir:
			image = cv2.imread(os.path.join(path,img))
			new_img = crop_img(image)
			new_img = cv2.resize(new_img,(IMG_SIZE,IMG_SIZE))
			if not os.path.exists(save_path):
				os.makedirs(save_path)
			cv2.imwrite(save_path+'/'+img, new_img)

In [8]:
# Code to load images borrowed from https://github.com/masoudnick/Brain-Tumor-MRI-Classification/blob/main/Brain_Tumor_Classification.ipynb
# We'll adjust this code to also load in the generated images.

# Change these to where your cleaned data are located
CLEANED_TRAINING_PATH = "C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\cleaned\\Training"
CLEANED_TESTING_PATH = "C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\cleaned\\Testing"


labels = ['glioma', 'meningioma', 'notumor', 'pituitary']

x_train = [] # training images.
y_train  = [] # training labels.
x_test_full = [] # test images.
y_test_full  = [] # test labels.
image_size = 200



for label in labels:
    trainPath = os.path.join(CLEANED_TRAINING_PATH,label)
    for file in tqdm(os.listdir(trainPath)):
        image = cv2.imread(os.path.join(trainPath, file),0) # load images in gray.
        image = cv2.bilateralFilter(image, 2, 50, 50) # remove images noise.
        image = cv2.applyColorMap(image, cv2.COLORMAP_BONE) # produce a pseudocolored image.
        # image = cv2.resize(image, (image_size, image_size)) # resize images into 200*200.
        x_train.append(image)
        y_train.append(labels.index(label))
    
    testPath = os.path.join(CLEANED_TESTING_PATH,label)
    for file in tqdm(os.listdir(testPath)):
        image = cv2.imread(os.path.join(testPath, file),0)
        image = cv2.bilateralFilter(image, 2, 50, 50)
        image = cv2.applyColorMap(image, cv2.COLORMAP_BONE)
        # image = cv2.resize(image, (image_size, image_size))
        x_test_full.append(image)
        y_test_full.append(labels.index(label))


        
# This will read in the generated images. No need to de-noise or resize them, since they were generated as 200*200. 
GEN_TRAIN_PATH = "C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\Generated\\Training"
GEN_TEST_PATH = "C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\Generated\\Testing"

x_train_fake = []
x_test_fake_full = []

for file in tqdm(os.listdir(GEN_TRAIN_PATH)):
    image = cv2.imread(os.path.join(GEN_TRAIN_PATH, file),0) # load images in gray.
    image = cv2.applyColorMap(image, cv2.COLORMAP_BONE) # produce a pseudocolored image.
    x_train_fake.append(image)
    
for file in tqdm(os.listdir(GEN_TEST_PATH)):
    image = cv2.imread(os.path.join(GEN_TEST_PATH, file),0) # load images in gray.
    image = cv2.applyColorMap(image, cv2.COLORMAP_BONE) # produce a pseudocolored image.
    x_test_fake_full.append(image)



# print(x_real.shape)
# print(x_fake.shape)

100%|██████████| 1000/1000 [00:01<00:00, 830.88it/s]


In [9]:
y_train = np.array(y_train, dtype=int)
y_test_full = np.array(y_test_full, dtype=int)

from sklearn.model_selection import train_test_split
x_train = np.array(x_train, dtype=np.float32) / 255.0 # normalize Images into range 0 to 1.
x_test_full = np.array(x_test_full, dtype=np.float32) / 255.0
x_valid, x_test, y_valid, y_test = train_test_split(x_test_full, y_test_full, test_size = 0.5)


# For now, we'll just keep the generated images separated into their training/testing groups.
# We'll give them labels y_train_fake and y_test_fake_full later by running them through
# the first discriminator we train.
x_train_fake = np.array(x_train_fake, dtype=np.float32) / 255.0
x_test_fake_full = np.array(x_test_fake_full, dtype=np.float32)/255.0

In [10]:
print(x_train_fake.shape)
print(x_test_fake_full.shape)

(5000, 200, 200, 3)
(1000, 200, 200, 3)


In [11]:
# Now we will implement the evaluation metric. The simplest is GQI: (1) Train a classifier on real images
#                                                                   (2) Use this classifier to get labels for generated images
#                                                                   (3) Train a similar classifier on generated images
#                                                                   (4) Determine accuracy ratio of the two classifiers.
#
# If possible, it would be better to calculate FID instead.

# tf.random.set_seed(42)
# np.random.seed(42)



# CNN for classifying the real images. Borrowed and adapted from HOA #7.
# We can use transfer learning to train the classifier for the fake images later.

from functools import partial

DefaultConv2D = partial(keras.layers.Conv2D,
                        kernel_size=3, activation='relu', padding="SAME")

BT_classify_real = keras.models.Sequential([
    DefaultConv2D(filters=50, kernel_size=7, input_shape=[200, 200, 3]),
    keras.layers.MaxPooling2D(pool_size=2),
    DefaultConv2D(filters=100),
    DefaultConv2D(filters=100),
    keras.layers.MaxPooling2D(pool_size=2),
    DefaultConv2D(filters=200),
    DefaultConv2D(filters=200),
    keras.layers.MaxPooling2D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dense(units=100, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=50, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=4, activation='softmax'),
])




# BT_classify_fake = keras.models.Sequential([
#     DefaultConv2D(filters=50, kernel_size=7, input_shape=[200, 200, 3]),
#     keras.layers.MaxPooling2D(pool_size=2),
#     DefaultConv2D(filters=100),
#     DefaultConv2D(filters=100),
#     keras.layers.MaxPooling2D(pool_size=2),
#     DefaultConv2D(filters=200),
#     DefaultConv2D(filters=200),
#     keras.layers.MaxPooling2D(pool_size=2),
#     keras.layers.Flatten(),
#     keras.layers.Dense(units=100, activation='relu'),
#     keras.layers.Dropout(0.5),
#     keras.layers.Dense(units=50, activation='relu'),
#     keras.layers.Dropout(0.5),
#     keras.layers.Dense(units=4, activation='softmax'),
# ])

In [12]:
BT_classify_real.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d (Conv2D)              (None, 200, 200, 50)      7400      
_________________________________________________________________
max_pooling2d (MaxPooling2D) (None, 100, 100, 50)      0         
_________________________________________________________________
conv2d_1 (Conv2D)            (None, 100, 100, 100)     45100     
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 100, 100, 100)     90100     
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 50, 50, 100)       0         
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 50, 50, 200)       180200    
_________________________________________________________________
conv2d_4 (Conv2D)            (None, 50, 50, 200)       3

In [13]:
# BT_classify_fake.summary()

In [14]:
# BT_classify_real.compile(loss = "sparse_categorical_crossentropy", optimizer = "rmsprop", metrics=["accuracy"])
# history_real = BT_classify_real.fit(x_train, y_train, validation_data = (x_valid, y_valid), epochs=10)

Epoch 1/10
179/179 [==============================] - 25s 100ms/step - loss: 3.2435 - accuracy: 0.3290 - val_loss: 1.2889 - val_accuracy: 0.3664
Epoch 2/10
179/179 [==============================] - 15s 86ms/step - loss: 1.1023 - accuracy: 0.5144 - val_loss: 0.8719 - val_accuracy: 0.6519
Epoch 3/10
179/179 [==============================] - 17s 97ms/step - loss: 0.8519 - accuracy: 0.6483 - val_loss: 0.6763 - val_accuracy: 0.7344
Epoch 4/10
179/179 [==============================] - 18s 98ms/step - loss: 0.7009 - accuracy: 0.7355 - val_loss: 0.6374 - val_accuracy: 0.7481
Epoch 5/10
179/179 [==============================] - 18s 102ms/step - loss: 0.5570 - accuracy: 0.7978 - val_loss: 0.6247 - val_accuracy: 0.7481
Epoch 6/10
179/179 [==============================] - 18s 100ms/step - loss: 0.4639 - accuracy: 0.8358 - val_loss: 0.5150 - val_accuracy: 0.7893
Epoch 7/10
179/179 [==============================] - 18s 99ms/step - loss: 0.4262 - accuracy: 0.8550 - val_loss: 0.5923 - val_accura

In [15]:
# BT_classify_real.save('C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\GAN eval\\BT_classify_real')

In [12]:
# Skip to here to load in the already-saved real classifier.
BT_classify_real = keras.models.load_model('C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\GAN eval\\BT_classify_real')

In [13]:
loss_real, acc_real = BT_classify_real.evaluate(x_test, y_test)

21/21 [==============================] - 4s 47ms/step - loss: 0.3384 - accuracy: 0.8704


In [14]:
# Use the classifier trained on the real images to classify the fake images.
# We need these labels to train a second classifier specialized to the fake images 
# so we can calculate the GQI.
#
# Again, because this is a large number of images, not all of them can be trained at the same time. 
# So we define a loop again to aid in getting the labels. 

train_fake_size = len(x_train_fake)
test_fake_size_full = len(x_test_fake_full)

y_train_fake = [] # np.zeros(train_fake_size)
y_test_fake_full = [] # np.zeros(test_fake_size_full)

# Batch size for how many images to classify at a given time
# Divisors of the list sizes are preferable.
num_img = 100

# Defining the number of laps to go over to cover the total number of images.
# We just need to divide the size of the train/test lists by the number of images we want.
# If num_img is a divisor of the sizes of train/test lists, then we can easily convert to int and get nice round numbers.
laps_train_fake = int(train_fake_size/num_img)
laps_test_fake_full = int(test_fake_size_full/num_img)


num_train = 0
num_test = 0

for lap in range(laps_train_fake):
    print("Obtaining generated labels number(s) %s to %s" % (num_img*lap, num_img*(lap + 1)-1))
    x_train_fake_subset = x_train_fake[num_img*lap:num_img*(lap + 1)]
    
    # Because the classifier uses categorical cross entropy, it returns 4 probabilities for each input.
    # We need to find the index of the largest probability, and this will serve as the label for the image.
    y_train_fake_subset = BT_classify_real.predict(x_train_fake_subset)
    labels = np.argmax(y_train_fake_subset, axis=1)
    y_train_fake = np.append(y_train_fake, labels)

for lap in range(laps_test_fake_full):
    print("Obtaining generated labels number(s) %s to %s" % (num_img*lap, num_img*(lap + 1)-1))
    x_test_fake_full_subset = x_test_fake_full[num_img*lap:num_img*(lap + 1)]
    
    y_test_fake_full_subset = BT_classify_real.predict(x_test_fake_full_subset)
    labels = np.argmax(y_test_fake_full_subset, axis=1)
    y_test_fake_full = np.append(y_test_fake_full, labels)

    
    
    


print(y_train_fake.shape)
print(y_test_fake_full.shape)

Obtaining generated labels number(s) 0 to 99
Obtaining generated labels number(s) 100 to 199
Obtaining generated labels number(s) 200 to 299
Obtaining generated labels number(s) 300 to 399
Obtaining generated labels number(s) 400 to 499
Obtaining generated labels number(s) 500 to 599
Obtaining generated labels number(s) 600 to 699
Obtaining generated labels number(s) 700 to 799
Obtaining generated labels number(s) 800 to 899
Obtaining generated labels number(s) 900 to 999
Obtaining generated labels number(s) 1000 to 1099
Obtaining generated labels number(s) 1100 to 1199
Obtaining generated labels number(s) 1200 to 1299
Obtaining generated labels number(s) 1300 to 1399
Obtaining generated labels number(s) 1400 to 1499
Obtaining generated labels number(s) 1500 to 1599
Obtaining generated labels number(s) 1600 to 1699
Obtaining generated labels number(s) 1700 to 1799
Obtaining generated labels number(s) 1800 to 1899
Obtaining generated labels number(s) 1900 to 1999
Obtaining generated lab

In [15]:
# To unload the previous model(s) and layers if we don't need them anymore (to make the most of memory limitations!)
tf.keras.backend.clear_session()

In [16]:
# print(np.max(y_train_fake))
# print(np.max(y_test_fake_full))
# Split the test data into valid/test:
x_valid_fake, x_test_fake, y_valid_fake, y_test_fake = train_test_split(x_test_full, y_test_full, test_size = 0.5)

In [17]:
# Build, train, and save a classifier for the fake samples using the labels generated above.

tf.keras.backend.clear_session()

BT_classify_fake = keras.models.Sequential([
    DefaultConv2D(filters=50, kernel_size=7, input_shape=[200, 200, 3]),
    keras.layers.MaxPooling2D(pool_size=2),
    DefaultConv2D(filters=100),
    DefaultConv2D(filters=100),
    keras.layers.MaxPooling2D(pool_size=2),
    DefaultConv2D(filters=200),
    DefaultConv2D(filters=200),
    keras.layers.MaxPooling2D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dense(units=100, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=50, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(units=4, activation='softmax'),
])

BT_classify_fake.compile(loss = "sparse_categorical_crossentropy", optimizer = "rmsprop", metrics=["accuracy"])
history_fake = BT_classify_real.fit(x_train_fake[0:2500], y_train_fake[0:2500], validation_data = (x_valid_fake[0:500], y_valid_fake[0:500]), epochs=10)

BT_classify_fake.save('C:\\Users\\Aihara\\Desktop\\Temporary Jupter Stuff\\GAN eval\\BT_classify_fake')

Epoch 1/10
79/79 [==============================] - 42s 121ms/step - loss: 1.0008 - accuracy: 0.7008 - val_loss: 0.8406 - val_accuracy: 0.6160
Epoch 2/10
79/79 [==============================] - 8s 103ms/step - loss: 0.6456 - accuracy: 0.7476 - val_loss: 0.6696 - val_accuracy: 0.7580
Epoch 3/10
79/79 [==============================] - 9s 110ms/step - loss: 0.5625 - accuracy: 0.7880 - val_loss: 1.4609 - val_accuracy: 0.6680
Epoch 4/10
79/79 [==============================] - 13s 159ms/step - loss: 0.4934 - accuracy: 0.8220 - val_loss: 1.1626 - val_accuracy: 0.6600
Epoch 5/10
79/79 [==============================] - 11s 134ms/step - loss: 0.4816 - accuracy: 0.8200 - val_loss: 2.9262 - val_accuracy: 0.6100
Epoch 6/10
79/79 [==============================] - 11s 143ms/step - loss: 0.4303 - accuracy: 0.8460 - val_loss: 1.4105 - val_accuracy: 0.6940
Epoch 7/10
79/79 [==============================] - 9s 119ms/step - loss: 0.4180 - accuracy: 0.8508 - val_loss: 4.6510 - val_accuracy: 0.5160
Ep

In [18]:
# I got CRAZY low validation accuracy. The classifier is probably overfitting.
# Interesting how the accuracy is so high to begin with, though. Not sure why it drops immediately.
# So either we need to find the optimal spot (like in one of the previous HOAs), or it's just an issue 
# with the generated images.
# I think it's mainly that the generated images from the DCGAN are sill pretty bad.
loss_cf, acc_cf = BT_classify_real.evaluate(x_test_fake, y_test_fake)

21/21 [==============================] - 1s 32ms/step - loss: 5.6534 - accuracy: 0.4558


In [21]:
# This is our GAN Quality Index result. 
# Not great, but it's a result.
GQI = (acc_cf/acc_real)*100.
GQI

52.36427436218525